# 1. Project Introduction

Welcome! In this notebook, we will explore **Naive Bayes**, a probabilistic classifier built on Bayes' Theorem.

### What is Naive Bayes?
* It is a **supervised learning** classification algorithm.
* It calculates the probability of each class given the input features and selects the class with the highest probability.
* It is called **"Naive"** because it assumes that all input features are independent of one another. For example, it assumes a discount word in an email is independent of the email length, which is rarely true but simplifies calculations massively.

### Why does it exist?
* It is incredibly fast, simple to implement, and performs exceptionally well on text classification and spam filtering.

### Real-World Use Cases:
* **Spam Filters**: Identifying spam emails.
* **Sentiment Analysis**: Classifying review text as positive or negative.


# 2. Problem Statement

* **Goal**: Predict whether an incoming email is **Spam (1)** or **Not Spam (0)** based on its length and keywords.
* **Business Value**: Protects email inbox users from unwanted spam messages.


In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn import metrics


# 4. Create Synthetic Dataset

We define email statistics for **100 emails**.
* **Email_Length**: Character count.
* **Contains_Discount_Word**: 1 if words like "free", "discount" are present, 0 otherwise.
* **Contains_Urgent_Word**: 1 if words like "now", "urgent" are present, 0 otherwise.
* **Is_Spam**: Target classification label.


In [ ]:
# Hardcoded email dataset
length = [
    50,  120, 30,  300, 45,  15,  180, 220, 60,  400, 20,  150, 70,  500, 80,  90,  320, 250, 110, 420,
    40,  130, 35,  280, 55,  25,  190, 240, 65,  380, 15,  160, 75,  480, 85,  95,  310, 260, 115, 410,
    45,  140, 38,  290, 50,  28,  200, 230, 70,  390, 18,  170, 72,  490, 88,  98,  330, 270, 120, 430,
    42,  135, 32,  295, 52,  22,  185, 235, 68,  385, 12,  165, 78,  475, 82,  92,  315, 255, 112, 405,
    300, 320, 280, 340, 410, 150, 250, 190, 370, 290, 180, 220, 450, 480, 500, 90,  100, 120, 200, 300
]

contains_discount = [
    0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1,
    0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1,
    0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1,
    0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1,
    1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1
]

contains_urgent = [
    0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1,
    0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1,
    0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1,
    0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1,
    1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1
]

is_spam = [
    0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1,
    0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1,
    0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1,
    0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1,
    1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1
]

df = pd.DataFrame({
    'Email_Length': length,
    'Contains_Discount_Word': contains_discount,
    'Contains_Urgent_Word': contains_urgent,
    'Is_Spam': is_spam
})

print("Dataset Shape:", df.shape)
print("First 5 rows:")
print(df.head())


# 5. Exploratory Data Analysis (EDA)


In [ ]:
# Chart 1: Email Length by Spam/Ham Class
plt.figure(figsize=(8, 4))
sns.boxplot(x='Is_Spam', y='Email_Length', data=df, palette='Pastel1')
plt.title('Email Length vs. Spam Class')
plt.xlabel('Is Spam (0 = Ham, 1 = Spam)')
plt.ylabel('Email Length (Characters)')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()


### What Did We Observe?
* Spam emails (class 1) generally have higher character counts than ham emails (class 0).


In [ ]:
# Chart 2: Spam occurrence count by Contains_Discount_Word
plt.figure(figsize=(6, 4))
sns.countplot(x='Contains_Discount_Word', hue='Is_Spam', data=df, palette='Set2')
plt.title('Discount Words vs. Spam Class')
# Custom labels for readable charts
plt.xticks([0, 1], ['No Discount Word', 'Contains Discount Word'])
plt.xlabel('')
plt.ylabel('Count')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.legend(['Not Spam', 'Spam'])
plt.show()


### What Did We Observe?
* The presence of a discount word correlates heavily with the Spam class. Very few non-spam emails contain discount words.


In [ ]:
# Data cleaning check
print("Null values count:", df.isnull().sum().sum())


In [ ]:
# Feature Selection
X = df[['Email_Length', 'Contains_Discount_Word', 'Contains_Urgent_Word']]
y = df['Is_Spam']


In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# 9. Model Building

* **How it works conceptually**: It applies Bayes' Theorem:
  $$P(	ext{Spam} | 	ext{Words}) = \frac{P(	ext{Words} | 	ext{Spam}) 	imes P(	ext{Spam})}{P(	ext{Words})}$$
  The algorithm multiplies the individual probability of each word occurring in a spam message, assuming word occurrences are independent.


In [ ]:
# Initialize Gaussian Naive Bayes model
model = GaussianNB()


In [ ]:
# Fit model
model.fit(X_train, y_train)


In [ ]:
# Predict spam outcomes
predictions = model.predict(X_test)


In [ ]:
# Compute classification evaluation metrics
accuracy = metrics.accuracy_score(y_test, predictions)
precision = metrics.precision_score(y_test, predictions)
recall = metrics.recall_score(y_test, predictions)
f1 = metrics.f1_score(y_test, predictions)
conf_matrix = metrics.confusion_matrix(y_test, predictions)

# Print metrics in plain English
print(f"Accuracy Score: {accuracy:.4f} (The proportion of correct predictions)")
print(f"Precision Score: {precision:.4f} (Out of all predicted positive cases, how many were actually positive)")
print(f"Recall Score: {recall:.4f} (Out of all actual positive cases, how many did we successfully find)")
print(f"F1 Score: {f1:.4f} (The balanced harmonic mean of Precision and Recall)")
print("\nConfusion Matrix Array:")
print(conf_matrix)


# 13. Visualizing Model Performance

We will display:
1. **Confusion Matrix Heatmap**.
2. **Feature Probability Density Curves**: Plotting distribution of email length for spam vs ham.


In [ ]:
# Plot 1: Confusion Matrix Heatmap
conf_matrix = metrics.confusion_matrix(y_test, predictions)
plt.figure(figsize=(6, 4))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Greens', 
            xticklabels=['Predicted Ham', 'Predicted Spam'], 
            yticklabels=['Actual Ham', 'Actual Spam'])
plt.title('Naive Bayes Confusion Matrix')
plt.show()


In [ ]:
# Plot 2: Probability Density Curves for Email_Length
plt.figure(figsize=(8, 4))
sns.kdeplot(df[df['Is_Spam'] == 0]['Email_Length'], label='Ham (Not Spam)', shade=True, color='blue')
sns.kdeplot(df[df['Is_Spam'] == 1]['Email_Length'], label='Spam', shade=True, color='red')
plt.title('Probability Density of Email Length')
plt.xlabel('Email Length')
plt.ylabel('Density Probability')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


### What Did We Observe?
* The peaks of the curves are distinct, indicating that email length is a useful factor for Gaussian probability estimation.


# 14. Model Interpretation

Naive Bayes calculates baseline prior probabilities:
* **Class Prior**: The overall probability of an email being spam ($P(	ext{Spam})$) vs. ham ($P(	ext{Ham})$) before reading its words.


In [ ]:
# Display class priors calculated by model
print("Class Priors (Ham, Spam):", model.class_prior_)
print("Class Counts:", model.class_count_)


# 15. Conclusion
* We built a spam classifier using email features.
* Probabilistic classifiers are fast and perform exceptionally well under independence assumptions.


# 16. Beginner ML Dictionary

Here are simple, one-sentence explanations of common Machine Learning terms to help you review:

* **Feature**: An input variable or column in your dataset used to make predictions (e.g., hours studied).
* **Target**: The output variable or label you want the model to predict (e.g., final exam score).
* **Training Data**: The portion of the dataset used to teach the model and find patterns.
* **Testing Data**: The portion of the dataset held back to evaluate how well the model performs on new, unseen data.
* **Prediction**: The output value generated by the trained model when given new input features.
* **Overfitting**: A scenario where the model learns the training data too well, including its noise, and performs poorly on new data.
* **Underfitting**: A scenario where the model is too simple to learn the underlying patterns in the training data, leading to poor performance on both training and test data.
* **Model**: The mathematical representation of the patterns learned from the training data by the algorithm.
* **Algorithm**: The set of rules or mathematical procedures followed to build the model from the data (e.g., Linear Regression).
* **Accuracy**: The percentage of correct predictions made by a classification model.
* **Cluster**: A group of similar data points grouped together by an unsupervised learning algorithm based on their characteristics.
* **Centroid**: The center point of a cluster, representing the average location of all data points belonging to that cluster.
